# Notebook 2 — Comparing and Classifying Ocean Sounds

**Self-guided activity · Unit 4: Identifying Ocean Sound Sources**

In this notebook, you will listen to and compare three hydroacoustic recordings:

1. **Humpback Whale Calls** — biological source
2. **Small Boat** — anthropogenic (human-made) source
3. **Ice Cracking in Cambridge Bay** — environmental source

For each recording, you will examine the waveform, **Fast Fourier Transform (FFT)** frequency spectrum and spectrogram. You will then describe the acoustic evidence, compare the recordings and explain how confidently you can classify them.

**No previous Python experience is required.** Read each explanation and run the code cells in order using **Shift + Enter**. When you see a question, examine the plot and try to answer before reading the suggested interpretation. You may change the display limits indicated in the code and run the cell again.

**Learning objectives:** By the end, you should be able to recognise tonal and broadband signals, distinguish continuous and impulsive sounds, identify repetition, and justify a source classification while acknowledging uncertainty.

## Important scientific notes — read before starting

**1. The recordings are MP3 audio files.** We analyse the audio supplied with this teaching package. MP3 compression and any processing applied when the recordings were prepared can affect the signal. The **44,100 Hz** sample rate read from these MP3 files is the rate of the *supplied files*, not necessarily the original hydrophone acquisition rate.

**2. These recordings are not calibrated to underwater pressure.** The waveform values are **relative digital amplitudes**, not pascals (Pa). Consequently, the graphs **must not be interpreted as sound pressure level (SPL, dB re 1 µPa)**. We also cannot use their waveform heights to decide which source was louder in the ocean.

**3. Frequency and time axes refer to the supplied MP3 playback.** Some educational marine audio files are sped up to make low-frequency signals audible. No confirmed speed-change factor is provided for these three files, so this notebook **does not convert playback frequencies or durations into original underwater values**. Do not assume that the 10× factor from Notebook 1 applies here.

**4. Spectrogram colour scales are relative.** Each spectrogram is expressed in decibels relative to its *own* maximum spectral power (0 dB). Brighter colours indicate stronger components **within that recording**; colours cannot be used to compare absolute acoustic levels between recordings.

**5. Source labels are supplied.** This is a **guided comparison**, not a blind classification test. Your task is to explain what acoustic features are consistent with each source, and whether other sources could produce similar features.

## Step 1 — Prepare Python and check your audio files

A **code cell** contains Python instructions; a **Markdown cell** contains explanations and questions. The code below imports our libraries. Run it once with **Shift + Enter**.

Put this notebook and all three MP3 files in the **same folder** before starting. If you see a `ModuleNotFoundError`, install the relevant package using the instructions in Unit 3. The required packages are **NumPy, Matplotlib, SciPy, librosa and IPython**.

In [ ]:
# Libraries for calculations, figures, audio loading and playback
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from scipy.signal import spectrogram
from IPython.display import Audio, display

# These filenames must match the MP3 files exactly.
whale_file = Path("Humpback Whale Calls (wav).mp3")
boat_file = Path("Small Boat (wav).mp3")
ice_file = Path("Ice Cracking in Cambridge Bay (wav).mp3")

for file_path in [whale_file, boat_file, ice_file]:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Cannot find {file_path.name}. Put the MP3 file in the same "
            "folder as the notebook, or check the filename."
        )

print("All three MP3 files were found. You can begin the analysis!")

### What should you look for?

- **Frequency:** How quickly the signal oscillates, measured in hertz (Hz).
- **Narrowband or tonal:** Energy concentrated near one or a few frequencies; often seen as thin horizontal or curved lines on a spectrogram.
- **Broadband:** Energy spread across a wide range of frequencies.
- **Continuous or impulsive:** A sustained sound versus a short, sudden event.
- **Repetition:** Do similar events repeat in time?
- **Frequency modulation:** Does a tone rise or fall in frequency?

**Three complementary views:** A **waveform** shows when and how strongly the signal changes; an **FFT spectrum** shows frequency content averaged across the whole selected recording; a **spectrogram** shows when different frequencies occur.

**Remember:** No single acoustic feature uniquely identifies a source.

---

## Recording 1 — Humpback Whale Calls

**Source category:** Biological

Whale calls may contain tones, frequency sweeps, and repeated sequences. Not every whale call is a song.

**Before looking at the figures:** This ONC recording contains humpback whale **non-song calls** from Folger Passage. Do you hear repeated low-pitched vocalisations or changes in pitch?

**Recording source:** Ocean Networks Canada (ONC). Full source URL: https://ibase.oceannetworks.ca/view-item?i=9860

Listen to the audio first and write down two characteristics you hear.

In [ ]:
# Load the whale MP3 without resampling its playback sampling frequency.
# librosa.load(..., sr=None) retains the sample rate of the MP3 file.
audio_whale, fs_whale = librosa.load(whale_file, sr=None, mono=True)

# Remove a constant offset (if present). This does not calibrate the data.
audio_whale = audio_whale - np.mean(audio_whale)
duration_whale = len(audio_whale) / fs_whale
nyquist_whale = fs_whale / 2

print("Recording: Humpback Whale Calls")
print(f"Playback sampling frequency: {fs_whale:,} Hz")
print(f"Playback duration: {duration_whale:.2f} s")
print(f"Number of audio samples: {len(audio_whale):,}")
print(f"Nyquist frequency of supplied MP3: {nyquist_whale:,.0f} Hz")

# Audio player: listen for patterns before analysing the plots.
display(Audio(filename=str(whale_file)))

### 1.1 — Waveform: when do events occur?

A **waveform** plots relative digital amplitude against **playback time**. It shows the timing and duration of sounds, but often cannot identify their frequencies directly.

`librosa.display.waveshow` displays a waveform efficiently, even for long recordings. It preserves the overall amplitude envelope without drawing millions of overlapping samples. We keep the complete signal for the FFT and spectrogram calculations.

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(audio_whale, sr=fs_whale)
plt.xlabel("Playback time (s)")
plt.ylabel("Relative digital amplitude")
plt.title("Humpback Whale Calls — Waveform")
plt.tight_layout()
plt.show()

### 1.2 — FFT: which frequencies are present?

The **Fast Fourier Transform (FFT)** calculates the frequency components of the **entire supplied recording**. Each frequency is represented by a peak or a region of spectral energy.

For easier interpretation, we divide FFT magnitudes by their maximum, so the largest plotted value is **1**. This is an **independently normalised, relative spectrum**, not a calibrated pressure spectrum. The normalisation lets us see the main peaks more clearly but does **not** make the absolute sound levels comparable between files.

In [ ]:
# Calculate a one-sided FFT because the audio samples are real-valued.
N_whale = len(audio_whale)
fft_whale = np.fft.rfft(audio_whale)
freq_whale = np.fft.rfftfreq(N_whale, d=1 / fs_whale)

# Relative FFT magnitude: 1 corresponds to the largest FFT magnitude.
fft_magnitude_whale = np.abs(fft_whale)
fft_relative_whale = fft_magnitude_whale / np.max(fft_magnitude_whale)

plt.figure(figsize=(12, 4))
plt.plot(freq_whale, fft_relative_whale, linewidth=0.8)
plt.xlim(0, 1000)   # Change this value to explore other frequency ranges.
plt.xlabel("Playback frequency (Hz)")
plt.ylabel("Relative FFT magnitude (maximum = 1)")
plt.title("Humpback Whale Calls — Frequency Spectrum")
plt.tight_layout()
plt.show()

### 1.3 — Spectrogram: when do frequencies occur?

A spectrogram performs many short spectral analyses to show **how frequency content varies through time**.

We use a window of **8,192 samples**, which provides a frequency-bin spacing of about **5.38 Hz** at 44,100 Hz, rather than the coarser 21.53 Hz of a 2,048-sample window. The trade-off is less precise timing of very brief events.

The colour scale is in **relative decibels**, with the maximum spectral power *within this recording's spectrogram* set to **0 dB**. Values below 0 dB indicate weaker components. **These are not dB re 1 µPa.**

In [ ]:
# Short-time power spectral density (PSD) using overlapping windows.
f_whale, t_whale, Sxx_whale = spectrogram(
    audio_whale,
    fs=fs_whale,
    nperseg=8192,
    noverlap=6144,
    scaling="density",
    mode="psd"
)

# Convert to decibels *relative to the maximum PSD in this recording*.
# This is NOT underwater SPL and must not be used to compare absolute levels.
Sxx_relative_whale = Sxx_whale / np.max(Sxx_whale)
Sxx_db_whale = 10 * np.log10(np.maximum(Sxx_relative_whale, 1e-10))

max_plot_frequency_whale = 1000  # Change this to zoom in/out.
frequency_mask_whale = f_whale <= max_plot_frequency_whale

plt.figure(figsize=(12, 5))
plt.pcolormesh(
    t_whale,
    f_whale[frequency_mask_whale],
    Sxx_db_whale[frequency_mask_whale, :],
    shading="auto",
    vmin=-70,
    vmax=0
)
plt.colorbar(label="Relative PSD (dB; maximum = 0 dB)")
plt.xlabel("Playback time (s)")
plt.ylabel("Playback frequency (Hz)")
plt.title("Humpback Whale Calls — Spectrogram")
plt.tight_layout()
plt.show()

### 1.4 — Interpret this recording

Try answering **before** expanding the suggested interpretation below.

1. At approximately which frequencies is the signal most prominent in the **supplied MP3**?
2. Is it predominantly **tonal, broadband, or a mixture**?
3. Is the sound continuous, intermittent or impulsive?
4. Can you see repeated patterns, changing frequencies or individual events?
5. Which representation (waveform, FFT, spectrogram) best supports your interpretation, and why?

<details>
<summary><strong>Suggested interpretation — check your observations</strong></summary>

Look for several separated low-frequency call events, some with tonal or changing-frequency structure. Strong energy in the supplied MP3 is concentrated mainly below a few hundred hertz, with notable components around 60–100 Hz. Some other frequencies and background sounds are also present. The calls are not perfectly identical, and the source cannot be proven from a single spectral feature.

The answer is qualitative. Exact peaks and timing estimates depend on the plotted range and how an event is defined. The known provider source label is supporting context, not proof that every component comes from that source.

</details>

---

## Recording 2 — Small Boat

**Source category:** Anthropogenic

Vessels can generate steady machinery tones and broadband noise associated with propellers and water flow. They may become louder or quieter during a pass.

**Before looking at the figures:** Listen for comparatively sustained machinery-like sound and any change in level as the boat passes or approaches.

**Recording source:** Ocean Networks Canada (ONC). Full source URL: https://www.oceannetworks.ca/multimedia/sounds/

Listen to the audio first and write down two characteristics you hear.

In [ ]:
# Load the boat MP3 without resampling its playback sampling frequency.
# librosa.load(..., sr=None) retains the sample rate of the MP3 file.
audio_boat, fs_boat = librosa.load(boat_file, sr=None, mono=True)

# Remove a constant offset (if present). This does not calibrate the data.
audio_boat = audio_boat - np.mean(audio_boat)
duration_boat = len(audio_boat) / fs_boat
nyquist_boat = fs_boat / 2

print("Recording: Small Boat")
print(f"Playback sampling frequency: {fs_boat:,} Hz")
print(f"Playback duration: {duration_boat:.2f} s")
print(f"Number of audio samples: {len(audio_boat):,}")
print(f"Nyquist frequency of supplied MP3: {nyquist_boat:,.0f} Hz")

# Audio player: listen for patterns before analysing the plots.
display(Audio(filename=str(boat_file)))

### 2.1 — Waveform: when do events occur?

A **waveform** plots relative digital amplitude against **playback time**. It shows the timing and duration of sounds, but often cannot identify their frequencies directly.

`librosa.display.waveshow` displays a waveform efficiently, even for long recordings. It preserves the overall amplitude envelope without drawing millions of overlapping samples. We keep the complete signal for the FFT and spectrogram calculations.

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(audio_boat, sr=fs_boat)
plt.xlabel("Playback time (s)")
plt.ylabel("Relative digital amplitude")
plt.title("Small Boat — Waveform")
plt.tight_layout()
plt.show()

### 2.2 — FFT: which frequencies are present?

The **Fast Fourier Transform (FFT)** calculates the frequency components of the **entire supplied recording**. Each frequency is represented by a peak or a region of spectral energy.

For easier interpretation, we divide FFT magnitudes by their maximum, so the largest plotted value is **1**. This is an **independently normalised, relative spectrum**, not a calibrated pressure spectrum. The normalisation lets us see the main peaks more clearly but does **not** make the absolute sound levels comparable between files.

In [ ]:
# Calculate a one-sided FFT because the audio samples are real-valued.
N_boat = len(audio_boat)
fft_boat = np.fft.rfft(audio_boat)
freq_boat = np.fft.rfftfreq(N_boat, d=1 / fs_boat)

# Relative FFT magnitude: 1 corresponds to the largest FFT magnitude.
fft_magnitude_boat = np.abs(fft_boat)
fft_relative_boat = fft_magnitude_boat / np.max(fft_magnitude_boat)

plt.figure(figsize=(12, 4))
plt.plot(freq_boat, fft_relative_boat, linewidth=0.8)
plt.xlim(0, 2500)   # Change this value to explore other frequency ranges.
plt.xlabel("Playback frequency (Hz)")
plt.ylabel("Relative FFT magnitude (maximum = 1)")
plt.title("Small Boat — Frequency Spectrum")
plt.tight_layout()
plt.show()

### 2.3 — Spectrogram: when do frequencies occur?

A spectrogram performs many short spectral analyses to show **how frequency content varies through time**.

We use a window of **8,192 samples**, which provides a frequency-bin spacing of about **5.38 Hz** at 44,100 Hz, rather than the coarser 21.53 Hz of a 2,048-sample window. The trade-off is less precise timing of very brief events.

The colour scale is in **relative decibels**, with the maximum spectral power *within this recording's spectrogram* set to **0 dB**. Values below 0 dB indicate weaker components. **These are not dB re 1 µPa.**

In [ ]:
# Short-time power spectral density (PSD) using overlapping windows.
f_boat, t_boat, Sxx_boat = spectrogram(
    audio_boat,
    fs=fs_boat,
    nperseg=8192,
    noverlap=6144,
    scaling="density",
    mode="psd"
)

# Convert to decibels *relative to the maximum PSD in this recording*.
# This is NOT underwater SPL and must not be used to compare absolute levels.
Sxx_relative_boat = Sxx_boat / np.max(Sxx_boat)
Sxx_db_boat = 10 * np.log10(np.maximum(Sxx_relative_boat, 1e-10))

max_plot_frequency_boat = 2500  # Change this to zoom in/out.
frequency_mask_boat = f_boat <= max_plot_frequency_boat

plt.figure(figsize=(12, 5))
plt.pcolormesh(
    t_boat,
    f_boat[frequency_mask_boat],
    Sxx_db_boat[frequency_mask_boat, :],
    shading="auto",
    vmin=-70,
    vmax=0
)
plt.colorbar(label="Relative PSD (dB; maximum = 0 dB)")
plt.xlabel("Playback time (s)")
plt.ylabel("Playback frequency (Hz)")
plt.title("Small Boat — Spectrogram")
plt.tight_layout()
plt.show()

### 2.4 — Interpret this recording

Try answering **before** expanding the suggested interpretation below.

1. At approximately which frequencies is the signal most prominent in the **supplied MP3**?
2. Is it predominantly **tonal, broadband, or a mixture**?
3. Is the sound continuous, intermittent or impulsive?
4. Can you see repeated patterns, changing frequencies or individual events?
5. Which representation (waveform, FFT, spectrogram) best supports your interpretation, and why?

<details>
<summary><strong>Suggested interpretation — check your observations</strong></summary>

The recording has a relatively sustained, broad noise background and multiple tonal components. Its amplitude and frequency distribution change as the vessel becomes more prominent. It is different from a sequence of isolated animal calls, but some frequency ranges overlap with the whale recording.

The answer is qualitative. Exact peaks and timing estimates depend on the plotted range and how an event is defined. The known provider source label is supporting context, not proof that every component comes from that source.

</details>

---

## Recording 3 — Ice Cracking in Cambridge Bay

**Source category:** Environmental

Ice cracking creates short, irregular pressure disturbances. Such transients may produce broadband energy visible as vertical features on spectrograms.

**Before looking at the figures:** Listen for short irregular crackling and popping events against the background.

**Recording source:** Ocean Networks Canada (ONC). Full source URL: https://ibase.oceannetworks.ca/view-item?i=9827

Listen to the audio first and write down two characteristics you hear.

In [ ]:
# Load the ice MP3 without resampling its playback sampling frequency.
# librosa.load(..., sr=None) retains the sample rate of the MP3 file.
audio_ice, fs_ice = librosa.load(ice_file, sr=None, mono=True)

# Remove a constant offset (if present). This does not calibrate the data.
audio_ice = audio_ice - np.mean(audio_ice)
duration_ice = len(audio_ice) / fs_ice
nyquist_ice = fs_ice / 2

print("Recording: Ice Cracking in Cambridge Bay")
print(f"Playback sampling frequency: {fs_ice:,} Hz")
print(f"Playback duration: {duration_ice:.2f} s")
print(f"Number of audio samples: {len(audio_ice):,}")
print(f"Nyquist frequency of supplied MP3: {nyquist_ice:,.0f} Hz")

# Audio player: listen for patterns before analysing the plots.
display(Audio(filename=str(ice_file)))

### 3.1 — Waveform: when do events occur?

A **waveform** plots relative digital amplitude against **playback time**. It shows the timing and duration of sounds, but often cannot identify their frequencies directly.

`librosa.display.waveshow` displays a waveform efficiently, even for long recordings. It preserves the overall amplitude envelope without drawing millions of overlapping samples. We keep the complete signal for the FFT and spectrogram calculations.

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(audio_ice, sr=fs_ice)
plt.xlabel("Playback time (s)")
plt.ylabel("Relative digital amplitude")
plt.title("Ice Cracking in Cambridge Bay — Waveform")
plt.tight_layout()
plt.show()

### 3.2 — FFT: which frequencies are present?

The **Fast Fourier Transform (FFT)** calculates the frequency components of the **entire supplied recording**. Each frequency is represented by a peak or a region of spectral energy.

For easier interpretation, we divide FFT magnitudes by their maximum, so the largest plotted value is **1**. This is an **independently normalised, relative spectrum**, not a calibrated pressure spectrum. The normalisation lets us see the main peaks more clearly but does **not** make the absolute sound levels comparable between files.

In [ ]:
# Calculate a one-sided FFT because the audio samples are real-valued.
N_ice = len(audio_ice)
fft_ice = np.fft.rfft(audio_ice)
freq_ice = np.fft.rfftfreq(N_ice, d=1 / fs_ice)

# Relative FFT magnitude: 1 corresponds to the largest FFT magnitude.
fft_magnitude_ice = np.abs(fft_ice)
fft_relative_ice = fft_magnitude_ice / np.max(fft_magnitude_ice)

plt.figure(figsize=(12, 4))
plt.plot(freq_ice, fft_relative_ice, linewidth=0.8)
plt.xlim(0, 5000)   # Change this value to explore other frequency ranges.
plt.xlabel("Playback frequency (Hz)")
plt.ylabel("Relative FFT magnitude (maximum = 1)")
plt.title("Ice Cracking in Cambridge Bay — Frequency Spectrum")
plt.tight_layout()
plt.show()

### 3.3 — Spectrogram: when do frequencies occur?

A spectrogram performs many short spectral analyses to show **how frequency content varies through time**.

We use a window of **8,192 samples**, which provides a frequency-bin spacing of about **5.38 Hz** at 44,100 Hz, rather than the coarser 21.53 Hz of a 2,048-sample window. The trade-off is less precise timing of very brief events.

The colour scale is in **relative decibels**, with the maximum spectral power *within this recording's spectrogram* set to **0 dB**. Values below 0 dB indicate weaker components. **These are not dB re 1 µPa.**

In [ ]:
# Short-time power spectral density (PSD) using overlapping windows.
f_ice, t_ice, Sxx_ice = spectrogram(
    audio_ice,
    fs=fs_ice,
    nperseg=8192,
    noverlap=6144,
    scaling="density",
    mode="psd"
)

# Convert to decibels *relative to the maximum PSD in this recording*.
# This is NOT underwater SPL and must not be used to compare absolute levels.
Sxx_relative_ice = Sxx_ice / np.max(Sxx_ice)
Sxx_db_ice = 10 * np.log10(np.maximum(Sxx_relative_ice, 1e-10))

max_plot_frequency_ice = 5000  # Change this to zoom in/out.
frequency_mask_ice = f_ice <= max_plot_frequency_ice

plt.figure(figsize=(12, 5))
plt.pcolormesh(
    t_ice,
    f_ice[frequency_mask_ice],
    Sxx_db_ice[frequency_mask_ice, :],
    shading="auto",
    vmin=-70,
    vmax=0
)
plt.colorbar(label="Relative PSD (dB; maximum = 0 dB)")
plt.xlabel("Playback time (s)")
plt.ylabel("Playback frequency (Hz)")
plt.title("Ice Cracking in Cambridge Bay — Spectrogram")
plt.tight_layout()
plt.show()

### 3.4 — Interpret this recording

Try answering **before** expanding the suggested interpretation below.

1. At approximately which frequencies is the signal most prominent in the **supplied MP3**?
2. Is it predominantly **tonal, broadband, or a mixture**?
3. Is the sound continuous, intermittent or impulsive?
4. Can you see repeated patterns, changing frequencies or individual events?
5. Which representation (waveform, FFT, spectrogram) best supports your interpretation, and why?

<details>
<summary><strong>Suggested interpretation — check your observations</strong></summary>

Several abrupt transient events are visible, with energy spread across a broad frequency range. They appear as short impulses in the waveform and as approximately vertical streaks in the spectrogram. The recording is intermittent, unlike more sustained boat noise; nevertheless, some biological or human-made clicks can also be broadband.

The answer is qualitative. Exact peaks and timing estimates depend on the plotted range and how an event is defined. The known provider source label is supporting context, not proof that every component comes from that source.

</details>

---

## Final activity — Compare and classify the three recordings

Fill in the table using what you **observed**, not just the source names. You may copy it into your notes and add approximate playback frequencies from your figures.

| Acoustic feature | Humpback whale | Small boat | Ice cracking |
|---|---|---|---|
| Source category | Biological | Anthropogenic | Environmental |
| Dominant **playback** frequency range | | | |
| Narrowband / tonal / broadband | | | |
| Continuous / intermittent / impulsive | | | |
| Repeated or changing patterns | | | |
| Strongest acoustic evidence for the classification | | | |
| An alternative possible source or limitation | | | |

### Final questions

1. Which source has the clearest repeated or frequency-modulated features?
2. Which source is most sustained through time?
3. Which source contains the clearest short broadband impulses?
4. Can biological and human-made sources overlap in frequency?
5. Which figure is most useful for a short event, and which helps identify frequency changes?
6. **Classification confidence:** For each file, assign **high**, **moderate** or **low** confidence *based only on the acoustic evidence*. Explain which observation supports your rating and what additional information would make you more confident.

**Confidence guide:** High = several independent features agree; moderate = some informative features, but alternatives remain plausible; low = weak or ambiguous evidence. These are qualitative judgements, **not** statistical probabilities.

<details>
<summary><strong>Final activity — suggested comparison and feedback</strong></summary>

| Acoustic feature | Humpback whale | Small boat | Ice cracking |
|---|---|---|---|
| Typical spectral character in these files | Prominent low-frequency tonal / modulated components | Sustained broadband energy plus tonal features | Broadly distributed energy during short events |
| Timing | Separated vocalisation-like events | More sustained and variable in level | Brief, irregular transients |
| Useful visual clue | Structured or repeated traces in spectrogram | Persistent bands and broad noise | Vertical broadband streaks and sharp waveform peaks |
| Important limitation | Other animals or equipment may produce similar tones | Other machinery or environmental noise may overlap | Animal clicks or impacts can also be impulsive |

1. **Repeated or modulated features:** The humpback example.
2. **Most sustained:** The small boat example.
3. **Short broadband impulses:** The ice-cracking example.
4. **Frequency overlap:** Yes. Overlapping frequencies do not imply the same source.
5. **Choosing a figure:** A waveform shows the timing of individual events; the FFT summarises overall frequency content; the spectrogram displays the timing *and* frequencies of events.
6. **Confidence:** More than one answer is defensible if supported by evidence. The recording's known source, deployment metadata and independent observations can strengthen a classification, but the plots alone rarely prove a unique source.

**Important:** The three audio files may have different gain or processing histories. Do not infer which physical source was louder from waveform height or normalised spectral colour.

</details>

## What you have learned

You have used Python to:

- Load and listen to underwater recordings;
- Identify acoustic timing and relative amplitude from waveforms;
- Identify prominent frequencies from FFT spectra;
- Interpret frequency changes and transient events from spectrograms;
- Compare source classes while acknowledging uncertainty.

**Scientific takeaway:** Reliable classification combines acoustic features, recording metadata and context. A single peak, transient or sound impression is seldom sufficient.

### Recording sources and reuse

All three supplied MP3 files are presented as examples from **Ocean Networks Canada (ONC)**. They are provided here as *MP3 playback versions* of media items originally labelled “(wav)”. Do not assume an MP3 retains the original hydrophone's full bandwidth or calibration.

- **Humpback Whale Calls (wav):** ONC, Folger Passage, hydrophone deployed at 95 m; ONC multimedia item dated 1 August 2012. https://ibase.oceannetworks.ca/view-item?i=9860
- **Ice Cracking in Cambridge Bay (wav):** ONC, Cambridge Bay, Nunavut, hydrophone deployed at 7.6 m; ONC multimedia item dated 9 January 2014. https://ibase.oceannetworks.ca/view-item?i=9827
- **Small Boat (wav):** File supplied with this activity, labelled as ONC audio. The exact ONC media record and original recording details should be confirmed before public release. ONC sounds catalogue: https://www.oceannetworks.ca/multimedia/sounds/

**Copyright:** The educational explanations and notebook code can be reused under the applicable course licence. Third-party audio remains subject to its original owner's terms. Public access to an ONC media item does not automatically establish permission to redistribute the MP3 under CC BY 4.0. Verify permissions before packaging the audio for EGU distribution.

**Practical troubleshooting:** If an MP3 cannot be found, check its filename and folder. If a library is missing, follow Unit 3's installation guide. If a plot looks empty, check whether its displayed frequency range includes the signal components and rerun the cell. If you rerun cells out of order and encounter an undefined variable, restart the kernel and execute cells from top to bottom.